# 📊 Exploratory Data Analysis (EDA)
## Marketing Intelligence AI Platform

**Purpose:** Understand the structure, quality, and statistical properties of every raw data source before modelling.

**Sections:**
1. Environment Setup
2. Load Raw Data Sources
3. Dataset Overview (shape, dtypes, nulls)
4. Univariate Analysis — Numeric Features
5. Univariate Analysis — Categorical Features
6. Bivariate / Correlation Analysis
7. Time-Series Trends
8. Missing Value Analysis
9. Outlier Detection
10. Channel-Level Comparison
11. Key Findings & Next Steps

---
## 1. Environment Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# Project imports
from shared.data_loader import DataLoader
from shared.constants import (
    DATE_COL, REVENUE_COL, SPEND_COL, CLICKS_COL,
    IMPRESSIONS_COL, CONVERSIONS_COL, CHANNEL_COL, CAMPAIGN_ID_COL
)

# Notebook display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

print('✅ Environment ready.')

---
## 2. Load Raw Data Sources

In [ ]:
# TODO: Uncomment once real CSV files are populated.

# google_ads  = DataLoader.load_google_ads()
# meta_ads    = DataLoader.load_meta_ads()
# microsoft   = DataLoader.load_microsoft_ads()
# ga4         = DataLoader.load_ga4()
# shopify     = DataLoader.load_shopify()
# campaign_md = DataLoader.load_campaign_metadata()

# --- Placeholder: synthetic data for notebook development ---
np.random.seed(42)
N = 1000

dates = pd.date_range('2023-01-01', periods=N, freq='D')
google_ads = pd.DataFrame({
    DATE_COL:         np.tile(dates[:N//4], 4)[:N],
    CAMPAIGN_ID_COL:  np.random.choice(['cg1','cg2','cg3'], N),
    IMPRESSIONS_COL:  np.random.randint(1000, 100000, N),
    CLICKS_COL:       np.random.randint(10, 5000, N),
    SPEND_COL:        np.random.uniform(50, 5000, N).round(2),
    CONVERSIONS_COL:  np.random.randint(0, 300, N),
    REVENUE_COL:      np.random.uniform(0, 20000, N).round(2),
    CHANNEL_COL:      'google_ads',
})

# Introduce some nulls and outliers for realistic EDA
google_ads.loc[google_ads.sample(frac=0.05).index, REVENUE_COL] = np.nan
google_ads.loc[google_ads.sample(frac=0.02).index, SPEND_COL] = google_ads[SPEND_COL].max() * 10

print(f'Google Ads shape: {google_ads.shape}')
google_ads.head()

---
## 3. Dataset Overview

In [ ]:
def dataset_overview(df: pd.DataFrame, name: str = 'Dataset') -> None:
    """Print a concise overview of a DataFrame."""
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(f'  Rows    : {df.shape[0]:,}')
    print(f'  Columns : {df.shape[1]}')
    print(f'  Memory  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
    print()
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    summary = pd.DataFrame({
        'dtype':     df.dtypes,
        'null_count': df.isnull().sum(),
        'null_pct':  null_pct,
        'unique':    df.nunique(),
    })
    display(summary)

dataset_overview(google_ads, 'Google Ads')

# TODO: Call dataset_overview() for each loaded source.
# dataset_overview(meta_ads,    'Meta Ads')
# dataset_overview(microsoft,   'Microsoft Ads')
# dataset_overview(ga4,         'GA4')
# dataset_overview(shopify,     'Shopify Orders')

---
## 4. Univariate Analysis — Numeric Features

In [ ]:
numeric_cols = google_ads.select_dtypes(include='number').columns.tolist()
print('Numeric columns:', numeric_cols)

# Descriptive statistics
google_ads[numeric_cols].describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols[:6]):
    ax = axes[i]
    sns.histplot(google_ads[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution: {col}')
    ax.set_xlabel('')

plt.suptitle('Google Ads — Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Univariate Analysis — Categorical Features

In [ ]:
cat_cols = google_ads.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)

for col in cat_cols:
    vc = google_ads[col].value_counts()
    fig = px.bar(x=vc.index, y=vc.values, labels={'x': col, 'y': 'count'},
                 title=f'Value Counts: {col}', color=vc.values,
                 color_continuous_scale='Blues')
    fig.show()

---
## 6. Bivariate / Correlation Analysis

In [ ]:
corr = google_ads[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Feature Correlation Heatmap — Google Ads',
    width=700, height=600,
)
fig.show()

# TODO: Add scatter matrix for top correlated feature pairs.

In [ ]:
# Revenue vs Spend scatter
fig = px.scatter(
    google_ads, x=SPEND_COL, y=REVENUE_COL,
    color=CAMPAIGN_ID_COL, opacity=0.6,
    trendline='ols',
    title='Revenue vs Spend by Campaign',
    labels={SPEND_COL: 'Spend ($)', REVENUE_COL: 'Revenue ($)'},
)
fig.show()

---
## 7. Time-Series Trends

In [ ]:
# TODO: Parse date column properly when loading real data.
ts = google_ads.copy()
ts[DATE_COL] = pd.to_datetime(ts[DATE_COL])
daily = ts.groupby(DATE_COL)[[REVENUE_COL, SPEND_COL, CLICKS_COL]].sum().reset_index()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=['Revenue', 'Spend', 'Clicks'])
for i, col in enumerate([REVENUE_COL, SPEND_COL, CLICKS_COL], start=1):
    fig.add_trace(go.Scatter(x=daily[DATE_COL], y=daily[col], mode='lines', name=col), row=i, col=1)

fig.update_layout(title='Daily Trends — Google Ads', height=700, showlegend=False)
fig.show()

---
## 8. Missing Value Analysis

In [ ]:
null_pct = (google_ads.isnull().sum() / len(google_ads) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if null_pct.empty:
    print('✅ No missing values found.')
else:
    fig = px.bar(
        x=null_pct.index, y=null_pct.values,
        labels={'x': 'Column', 'y': '% Missing'},
        title='Missing Value Percentage by Column',
        color=null_pct.values, color_continuous_scale='Reds',
    )
    fig.show()
    print(null_pct)

---
## 9. Outlier Detection

In [ ]:
# Box plots for outlier visualisation
fig = px.box(
    google_ads.melt(value_vars=numeric_cols),
    x='variable', y='value',
    title='Box Plots — Numeric Features (Google Ads)',
    points='outliers', color='variable',
)
fig.update_layout(showlegend=False)
fig.show()

# TODO: Apply IQR-based outlier flagging and log outlier counts per column.

---
## 10. Channel-Level Comparison

**TODO:** Load all channels and compare key metrics (Revenue, ROAS, CTR, CPC) side-by-side.

In [ ]:
# TODO: Merge all channel DataFrames and run cross-channel comparison.
# channels = pd.concat([google_ads, meta_ads, microsoft], ignore_index=True)
# fig = px.box(channels, x=CHANNEL_COL, y=REVENUE_COL, color=CHANNEL_COL,
#              title='Revenue Distribution by Channel')
# fig.show()

print('TODO: Implement cross-channel comparison once all sources are loaded.')

---
## 11. Key Findings & Next Steps

**Findings:**
- [ ] TODO: Document key distribution insights here.
- [ ] TODO: Note columns with high null rates.
- [ ] TODO: Identify outlier-prone columns.
- [ ] TODO: Record strongest feature correlations with Revenue.

**Next Steps:**
1. Open `FeatureEngineering.ipynb` to engineer derived features.
2. Open `ModelTraining.ipynb` after feature engineering is complete.
3. Update `shared/constants.py` with final feature column lists.